In [3]:
import pandas as pd
import folium
import geopandas as gpd

In [4]:
pop_df = pd.read_csv('Canada Population 2021 Data.csv')
pop_df = pop_df[pop_df['Location'] != 'Canada'].copy()
pop_df.head()


,Location,Non_Indigenous,First_Nations,Inuit,Métis
1,Newfoundland and Labrador,443310,40415,10245,6715
2,Prince Edward Island,144930,3755,255,915
3,Nova Scotia,890045,40715,1860,18310
4,New Brunswick,712025,31850,1025,9445
5,Quebec,13511270,288525,18025,44275


In [5]:

pop_df['Location'] = pop_df['Location'].str.replace('Yukon', 'Yukon Territory')

indig_total = pop_df['First_Nations'] + pop_df['Inuit'] + pop_df['Métis']
pop_df['Total_Pop'] = indig_total + pop_df['Non_Indigenous']
pop_df['pct_indigenous'] = (indig_total / pop_df['Total_Pop']) * 100
pop_df['pct_str'] = pop_df['pct_indigenous'].round(0).astype(int).astype(str) + '%'

In [6]:

gdf = gpd.read_file('Canada Provinces Data.json')
pop_df['Location'] = pop_df['Location'].str.replace('Yukon', 'Yukon Territory')
merged = gdf.merge(pop_df, left_on='name', right_on='Location', how='left')

merged = merged[~merged['geometry'].isna()]

merged = merged.select_dtypes(include=['number', 'object', 'geometry'])

merged.head()

,name,cartodb_id,geometry,Location,Non_Indigenous,First_Nations,Inuit,Métis,Total_Pop,pct_indigenous,pct_str
0,Quebec,1,"MULTIPOLYGON (((-78.01917 62.59193, -77.86722 ...",Quebec,13511270.0,288525.0,18025.0,44275.0,13862095.0,2.530822,3%
1,Newfoundland and Labrador,5,"MULTIPOLYGON (((-55.88306 53.48638, -55.72944 ...",Newfoundland and Labrador,443310.0,40415.0,10245.0,6715.0,500685.0,11.459301,11%
2,British Columbia,6,"MULTIPOLYGON (((-131.0771 52.15009, -130.9481 ...",British Columbia,4604060.0,209320.0,2340.0,85205.0,4900925.0,6.057326,6%
3,Nunavut,12,"MULTIPOLYGON (((-109.97841 78.67106, -109.4053...",Nunavut,5250.0,610.0,30855.0,125.0,36840.0,85.749186,86%
4,Northwest Territories,13,"MULTIPOLYGON (((-110.3944 78.75221, -109.97841...",Northwest Territories,20280.0,14005.0,4680.0,2525.0,41490.0,51.120752,51%


In [9]:
CANADA_CENTER = [62.24, -96.28]
m = folium.Map(location=CANADA_CENTER, zoom_start=3, tiles=None)

layer_configs = [
    {'name': 'Percent Indigenous', 'col': 'pct_indigenous', 'color': 'YlGn'},
    {'name': 'First Nations', 'col': 'First_Nations', 'color': 'OrRd'},
    {'name': 'Inuit', 'col': 'Inuit', 'color': 'PuBu'},
    {'name': 'Métis', 'col': 'Métis', 'color': 'BuPu'}
]

for cfg in layer_configs:

    fg = folium.FeatureGroup(name=cfg['name']).add_to(m)

    cp = folium.Choropleth(
        geo_data='Canada Provinces Data.json',
        data=merged,
        columns=['Location', cfg['col']],
        key_on='feature.properties.name',
        fill_color=cfg['color'],
        fill_opacity=0.7,
        line_opacity=0.4,
        highlight=True,
        name=cfg['name']
    )
    
    cp.add_to(fg)
    cp.add_to(m)

   
    folium.features.GeoJson(
        data=merged,
        style_function=lambda x: {'color':'black','fillColor':'transparent','weight':0.5},
        tooltip=folium.features.GeoJsonTooltip(
            fields=['Location', 'pct_str', 'Total_Pop', 'First_Nations', 'Inuit', 'Métis'],
            aliases=['Province/Territory:', 'Percent Indigenous:', 'Total Population:',
                     'First Nations:', 'Inuit:', 'Métis:'],
            localize=True,
            style='''
                background-color: #F0EFEF;
                border: 2px solid black;
                border-radius: 3px;
                box-shadow: 3px;
                font-size: medium;
            '''
        )
    ).add_to(fg)

In [10]:
folium.LayerControl(collapsed=False).add_to(m)
title_html = '<h3 align="center" style="font-size:20px;color:black"><b>Distribution of Indigenous Populations in Canada (2021)</b></h3>'
m.get_root().html.add_child(folium.Element(title_html))
m.save('map.html')

import webbrowser
webbrowser.open('map.html')



True

In [ ]:
import pandas as pd
import plotly.express as px


In [12]:
df = pd.read_csv('Canada Merge Data.csv')

In [34]:
df = df[df['Domain'] != 'Missing'].copy()
df['Domain'] = df['Domain'].replace('NonIndigenous [reference]', 'Non‑Indigenous')

In [36]:
df = df.dropna(subset=['CR_PERC', 'CR_FI', 'Geography', 'Sex'])
print(f"Rows: {len(df)}")

Rows: 87


In [37]:
x_min = df['CR_PERC'].min()
x_max = df['CR_PERC'].max()
y_min = df['CR_FI'].min()
y_max = df['CR_FI'].max()
x_pad = (x_max - x_min) * 0.05
y_pad = (y_max - y_min) * 0.05

In [ ]:
fig = px.scatter(
    df,
    x='CR_PERC',
    y='CR_FI',
    color='Domain',
    symbol='Domain',
    animation_frame='Sex',
    hover_name='Geography',
    hover_data={
        'CR_PERC': ':.2f',        
        'CR_FI': ':.2f',
        'DENOM_PERC': ':,.0f',      
        'DENOM_FI': ':,.0f',
        'Domain': False,
        'Geography': False
    },
    labels={
        'CR_PERC': 'Percentage Reporting Fair or Poor Mental Health',
        'CR_FI': 'Percentage with Food Insecurity',
        'Domain': 'Population Group'
    },
    title='Mental Health vs Food Insecurity by Sex',
    width=900,
    height=600,
    range_x=[x_min - x_pad, x_max + x_pad],
    range_y=[y_min - y_pad, y_max + y_pad]
)

In [ ]:
fig.update_traces(marker=dict(color='gray'), selector={'name': 'Non‑Indigenous'})

indigenous_groups = ['First Nations off reserve', 'Inuit', 'Métis']
colors = px.colors.qualitative.Set1
for i, group in enumerate(indigenous_groups):
    if group in df['Domain'].unique():
        fig.update_traces(marker=dict(color=colors[i % len(colors)]), selector={'name': group})

In [ ]:
fig.update_layout(
    title={
        'text': 'Mental Health vs Food Insecurity<br><sup>By Sex and Population Group</sup>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    legend_title_text='Population Group',
    legend_font_size=12,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    template='plotly_white'   
)

In [41]:
fig.write_image('scatter.png')
fig.write_html('scatter.html')
print("Scatterplot saved as 'scatter.png' and 'scatter.html'")

Scatterplot saved as 'scatter.png' and 'scatter.html'
